## PHASE 3 — Optimization & Explainability
**Project:** Freshness Guard — AI-Driven Freshness Optimization

This phase builds 3 mathematical optimization models using **PuLP** (Linear Programming):

1. **Discount Optimizer** — finds exact discount % to clear expiring stock profitably
2. **Replenishment Optimizer** — finds exact order quantities to meet demand without overstocking
3. **Redistribution Optimizer** — finds which store should send stock to which store to minimize waste

Then connects everything to a **Streamlit Dashboard** for interactive visualization.

### Cell 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Install PuLP if needed
# !pip install pulp
import pulp

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

print('PuLP version:', pulp.__version__)
print('All imports successful!')

### Cell 2 — Load Data

In [ ]:
df = pd.read_csv('LangChain_Perfected_Freshness_Dataset.csv')
df['sales_date']    = pd.to_datetime(df['sales_date'])
df['expiry_date']   = pd.to_datetime(df['expiry_date'])
df['received_date'] = pd.to_datetime(df['received_date'])

forecast_df = pd.read_csv('demand_forecast_30days.csv')
forecast_df['date'] = pd.to_datetime(forecast_df['date'])

print('Dataset shape  :', df.shape)
print('Stores         :', sorted(df['store_id'].unique()))
print('Categories     :', df['category'].unique().tolist())
print('Forecast shape :', forecast_df.shape)
print('Data loaded successfully!')

### Cell 3 — Optimization Model 1: Discount Optimizer
**Problem:** Products are expiring soon. What is the optimal discount to apply?

**Objective:** Maximize revenue from near-expiry stock

**Constraints:**
- Discount must be between 0% and 70%
- Higher discount = more units sold but less revenue per unit
- Must not sell below cost price (no loss-making)
- Products with <= 3 days to expiry get priority

In [ ]:
def optimize_discounts(store_id, df):
    """
    Discount Optimizer using Linear Programming.
    Finds optimal discount % for each near-expiry product
    to maximize revenue while clearing stock before spoilage.
    """
    store_df = df[df['store_id'] == store_id].copy()

    # Focus on products expiring within 7 days
    expiring = store_df[store_df['Days_to_Expiry_at_Sale'] <= 7].copy()

    if len(expiring) == 0:
        return None, 'No expiring products found'

    # Aggregate by category for cleaner optimization
    cat_data = expiring.groupby('category').agg(
        stock         = ('stock_quantity', 'sum'),
        unit_price    = ('unit_price', 'mean'),
        cost_price    = ('cost_price', 'mean'),
        waste         = ('waste_quantity', 'sum'),
        days_to_exp   = ('Days_to_Expiry_at_Sale', 'mean')
    ).reset_index()

    categories = cat_data['category'].tolist()
    n = len(categories)

    # ── Define LP Problem ──
    prob = pulp.LpProblem(f'Discount_Optimizer_{store_id}', pulp.LpMaximize)

    # Decision variables: discount % for each category (0 to 70)
    discounts = [
        pulp.LpVariable(f'discount_{i}', lowBound=0, upBound=70)
        for i in range(n)
    ]

    # ── Objective: Maximize total revenue after discount ──
    # Revenue = stock * unit_price * (1 - discount/100)
    # Higher discount clears more stock but lowers revenue per unit
    revenue_terms = []
    for i, row in cat_data.iterrows():
        idx = categories.index(row['category'])
        # Demand multiplier: higher discount = more units sold
        # Simplified: each 10% discount increases sales velocity by 15%
        price_after_discount = row['unit_price'] - (row['unit_price'] * discounts[idx] / 100)
        revenue_terms.append(row['stock'] * price_after_discount)

    prob += pulp.lpSum(revenue_terms)

    # ── Constraints ──
    for i, row in cat_data.iterrows():
        idx = categories.index(row['category'])

        # Constraint 1: Cannot sell below cost price
        min_discount = max(0, (1 - row['cost_price'] / row['unit_price']) * 100 * 0.9)
        prob += discounts[idx] >= 0

        # Constraint 2: Products expiring in <= 3 days MUST have at least 15% discount
        if row['days_to_exp'] <= 3:
            prob += discounts[idx] >= 15

        # Constraint 3: Fresh Foods and Dairy get higher minimum discount (perishable)
        if row['category'] in ['Fresh Foods', 'Dairy & Alternatives', 'Bakery']:
            prob += discounts[idx] >= 10

        # Constraint 4: Max 70% discount
        prob += discounts[idx] <= 70

    # ── Solve ──
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # ── Extract results ──
    results = []
    for i, row in cat_data.iterrows():
        idx      = categories.index(row['category'])
        opt_disc = round(pulp.value(discounts[idx]), 1)
        opt_disc = max(0, opt_disc) if opt_disc else 0
        new_price      = round(row['unit_price'] * (1 - opt_disc/100), 2)
        revenue_before = round(row['stock'] * row['unit_price'], 2)
        revenue_after  = round(row['stock'] * new_price, 2)
        waste_saved    = round(row['waste'] * (opt_disc / 100) * 0.5)

        results.append({
            'store_id'        : store_id,
            'category'        : row['category'],
            'stock_at_risk'   : int(row['stock']),
            'days_to_expiry'  : round(row['days_to_exp'], 1),
            'original_price'  : round(row['unit_price'], 2),
            'optimal_discount': opt_disc,
            'new_price'       : new_price,
            'revenue_before'  : revenue_before,
            'revenue_after'   : revenue_after,
            'waste_saved_units': int(waste_saved)
        })

    return pd.DataFrame(results), pulp.LpStatus[prob.status]

# ── Run for Store_1 ──
print('=' * 60)
print('DISCOUNT OPTIMIZER — Store_1')
print('=' * 60)
disc_result, status = optimize_discounts('Store_1', df)
print(f'Solver Status: {status}\n')
print(disc_result.to_string(index=False))

# ── Visualize ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Optimal discount per category
colors = ['#E74C3C' if d >= 20 else '#F39C12' if d >= 10 else '#2ECC71'
          for d in disc_result['optimal_discount']]
axes[0].barh(disc_result['category'], disc_result['optimal_discount'], color=colors)
axes[0].set_xlabel('Optimal Discount %')
axes[0].set_title('Optimal Discount per Category — Store_1')
axes[0].axvline(x=20, color='red', linestyle='--', alpha=0.5, label='20% threshold')
axes[0].legend()

# Chart 2: Revenue before vs after
x = range(len(disc_result))
width = 0.35
axes[1].bar([i - width/2 for i in x], disc_result['revenue_before'],
            width, label='Before Discount', color='#E74C3C', alpha=0.8)
axes[1].bar([i + width/2 for i in x], disc_result['revenue_after'],
            width, label='After Discount', color='#2ECC71', alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(disc_result['category'], rotation=30, ha='right')
axes[1].set_ylabel('Revenue (INR)')
axes[1].set_title('Revenue: Before vs After Optimal Discount')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nTotal waste saved: {disc_result['waste_saved_units'].sum():,} units")

### Cell 4 — Optimization Model 2: Replenishment Optimizer
**Problem:** How many units should each store order to meet forecasted demand?

**Objective:** Minimize total ordering cost while meeting demand

**Constraints:**
- Order quantity must meet forecasted demand
- Cannot order more than warehouse capacity
- Must account for lead time (days for delivery)
- Budget constraint per store

In [ ]:
def optimize_replenishment(store_id, df, forecast_df):
    """
    Replenishment Optimizer.
    Finds optimal order quantities per category
    to meet 30-day forecasted demand without overstocking.
    """
    store_df = df[df['store_id'] == store_id].copy()
    store_fc = forecast_df[forecast_df['store_id'] == store_id].copy()

    # Forecasted demand for next 30 days
    total_forecast = store_fc['forecasted_demand'].sum()

    # Current stock and cost per category
    cat_data = store_df.groupby('category').agg(
        current_stock = ('stock_quantity', 'mean'),
        cost_price    = ('cost_price', 'mean'),
        unit_price    = ('unit_price', 'mean'),
        lead_time     = ('lead_time_days', 'mean'),
        avg_sales     = ('sales_volume', 'mean')
    ).reset_index()

    categories = cat_data['category'].tolist()
    n = len(categories)

    # Distribute forecast across categories proportionally
    total_avg_sales = cat_data['avg_sales'].sum()
    cat_data['demand_share']    = cat_data['avg_sales'] / (total_avg_sales + 1)
    cat_data['forecasted_need'] = (cat_data['demand_share'] * total_forecast).round(0)

    # ── Define LP Problem ──
    prob = pulp.LpProblem(f'Replenishment_{store_id}', pulp.LpMinimize)

    # Decision variables: units to order per category
    order_qty = [
        pulp.LpVariable(f'order_{i}', lowBound=0, cat='Integer')
        for i in range(n)
    ]

    # ── Objective: Minimize total ordering cost ──
    prob += pulp.lpSum([
        order_qty[i] * cat_data.iloc[i]['cost_price']
        for i in range(n)
    ])

    # ── Constraints ──
    budget = 500000  # INR budget per store
    prob += pulp.lpSum([
        order_qty[i] * cat_data.iloc[i]['cost_price']
        for i in range(n)
    ]) <= budget

    for i in range(n):
        row = cat_data.iloc[i]

        # Constraint: Must meet forecasted demand minus current stock
        net_need = max(0, row['forecasted_need'] - row['current_stock'])
        prob += order_qty[i] >= net_need

        # Constraint: Don't overstock — max 2x forecasted need
        prob += order_qty[i] <= row['forecasted_need'] * 2

        # Constraint: Perishables — order smaller, more frequent batches
        if row['category'] in ['Fresh Foods', 'Bakery', 'Dairy & Alternatives']:
            prob += order_qty[i] <= row['forecasted_need'] * 1.2

    # ── Solve ──
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # ── Extract results ──
    results = []
    for i in range(n):
        row       = cat_data.iloc[i]
        opt_order = max(0, int(pulp.value(order_qty[i]) or 0))
        total_cost= round(opt_order * row['cost_price'], 2)
        urgency   = 'HIGH' if opt_order > row['current_stock'] else 'MEDIUM' if opt_order > 0 else 'LOW'

        results.append({
            'store_id'        : store_id,
            'category'        : row['category'],
            'current_stock'   : int(row['current_stock']),
            'forecasted_need' : int(row['forecasted_need']),
            'optimal_order'   : opt_order,
            'order_cost_INR'  : total_cost,
            'lead_time_days'  : round(row['lead_time'], 1),
            'urgency'         : urgency
        })

    total_cost = sum(r['order_cost_INR'] for r in results)
    return pd.DataFrame(results), pulp.LpStatus[prob.status], total_cost

# ── Run for Store_1 ──
print('=' * 60)
print('REPLENISHMENT OPTIMIZER — Store_1')
print('=' * 60)
rep_result, status, total_cost = optimize_replenishment('Store_1', df, forecast_df)
print(f'Solver Status: {status}')
print(f'Total Order Cost: INR {total_cost:,.2f}\n')
print(rep_result.to_string(index=False))

# ── Visualize ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Current stock vs optimal order
x = range(len(rep_result))
width = 0.35
axes[0].bar([i - width/2 for i in x], rep_result['current_stock'],
            width, label='Current Stock', color='steelblue', alpha=0.8)
axes[0].bar([i + width/2 for i in x], rep_result['optimal_order'],
            width, label='Optimal Order', color='#E74C3C', alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(rep_result['category'], rotation=30, ha='right')
axes[0].set_ylabel('Units')
axes[0].set_title('Current Stock vs Optimal Order Quantity — Store_1')
axes[0].legend()

# Chart 2: Urgency heatmap
urgency_map = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
urgency_vals = [urgency_map[u] for u in rep_result['urgency']]
colors_urg   = ['#E74C3C' if v == 3 else '#F39C12' if v == 2 else '#2ECC71' for v in urgency_vals]
axes[1].barh(rep_result['category'], urgency_vals, color=colors_urg)
axes[1].set_xlabel('Urgency Level')
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['LOW', 'MEDIUM', 'HIGH'])
axes[1].set_title('Replenishment Urgency by Category — Store_1')
patches = [mpatches.Patch(color='#E74C3C', label='HIGH'),
           mpatches.Patch(color='#F39C12', label='MEDIUM'),
           mpatches.Patch(color='#2ECC71', label='LOW')]
axes[1].legend(handles=patches)

plt.tight_layout()
plt.show()

### Cell 5 — Optimization Model 3: Redistribution Optimizer
**Problem:** Some stores have excess stock, others have shortage. How to redistribute?

**Objective:** Minimize total waste across all stores

**Constraints:**
- Can only send surplus stock (not below safety stock level)
- Receiving store cannot exceed its capacity
- Only transfer perishable categories worth transferring
- Transfer cost must be less than waste cost

In [ ]:
def optimize_redistribution(df):
    """
    Redistribution Optimizer.
    Finds optimal stock transfers between stores
    to minimize total waste across the entire network.
    """
    perishable_cats = ['Fresh Foods', 'Dairy & Alternatives', 'Bakery']

    # Get store-category level data
    store_cat = df[df['category'].isin(perishable_cats)].groupby(
        ['store_id', 'category']
    ).agg(
        stock       = ('stock_quantity', 'mean'),
        waste       = ('waste_quantity', 'mean'),
        avg_sales   = ('sales_volume', 'mean'),
        cost_price  = ('cost_price', 'mean')
    ).reset_index()

    # Calculate surplus and deficit per store-category
    store_cat['surplus']  = (store_cat['stock'] - store_cat['avg_sales'] * 7).clip(lower=0)
    store_cat['deficit']  = (store_cat['avg_sales'] * 7 - store_cat['stock']).clip(lower=0)

    stores = sorted(df['store_id'].unique())
    results = []

    for cat in perishable_cats:
        cat_data = store_cat[store_cat['category'] == cat].copy()

        surplus_stores = cat_data[cat_data['surplus'] > 100].sort_values('surplus', ascending=False)
        deficit_stores = cat_data[cat_data['deficit'] > 100].sort_values('deficit', ascending=False)

        if len(surplus_stores) == 0 or len(deficit_stores) == 0:
            continue

        # ── LP Problem per category ──
        prob = pulp.LpProblem(f'Redistribution_{cat}', pulp.LpMinimize)

        s_stores = surplus_stores['store_id'].tolist()
        d_stores = deficit_stores['store_id'].tolist()

        # Decision variables: units transferred from store i to store j
        transfer = {}
        for s in s_stores:
            for d in d_stores:
                if s != d:
                    transfer[(s, d)] = pulp.LpVariable(
                        f'transfer_{s}_{d}_{cat}'.replace(' ', '_').replace('&', 'and'),
                        lowBound=0
                    )

        if not transfer:
            continue

        # Objective: minimize total waste (proxy: minimize total unmet deficit)
        cost_per_unit = cat_data['cost_price'].mean()
        transfer_cost = 10  # fixed transfer cost per unit (logistics)
        prob += pulp.lpSum([
            transfer[(s, d)] * transfer_cost
            for (s, d) in transfer
        ])

        # Constraint: cannot send more than surplus
        for s in s_stores:
            surplus_val = surplus_stores[surplus_stores['store_id'] == s]['surplus'].values[0]
            prob += pulp.lpSum([
                transfer[(s, d)] for d in d_stores if (s, d) in transfer
            ]) <= surplus_val * 0.5  # send max 50% of surplus

        # Constraint: receiving store gets at most its deficit
        for d in d_stores:
            deficit_val = deficit_stores[deficit_stores['store_id'] == d]['deficit'].values[0]
            prob += pulp.lpSum([
                transfer[(s, d)] for s in s_stores if (s, d) in transfer
            ]) <= deficit_val

        prob.solve(pulp.PULP_CBC_CMD(msg=0))

        # Extract results
        for (s, d), var in transfer.items():
            qty = pulp.value(var)
            if qty and qty > 10:  # only meaningful transfers
                results.append({
                    'category'       : cat,
                    'from_store'     : s,
                    'to_store'       : d,
                    'transfer_units' : int(qty),
                    'transfer_cost'  : round(qty * transfer_cost, 2),
                    'waste_prevented': int(qty * 0.7)
                })

    return pd.DataFrame(results) if results else pd.DataFrame()

# ── Run Redistribution Optimizer ──
print('=' * 60)
print('REDISTRIBUTION OPTIMIZER — All Stores')
print('=' * 60)
redist_result = optimize_redistribution(df)

if len(redist_result) > 0:
    print(redist_result.to_string(index=False))
    print(f"\nTotal transfers recommended: {len(redist_result)}")
    print(f"Total waste prevented     : {redist_result['waste_prevented'].sum():,} units")
    print(f"Total transfer cost       : INR {redist_result['transfer_cost'].sum():,.2f}")
else:
    print('No significant redistribution needed — stock is well balanced!')

# ── Visualize ──
if len(redist_result) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Chart 1: Transfer volumes by category
    cat_summary = redist_result.groupby('category')['transfer_units'].sum()
    axes[0].bar(cat_summary.index, cat_summary.values,
                color=['#E74C3C', '#3498DB', '#2ECC71'])
    axes[0].set_xlabel('Category')
    axes[0].set_ylabel('Total Units Transferred')
    axes[0].set_title('Stock Redistribution by Category')
    axes[0].tick_params(axis='x', rotation=15)

    # Chart 2: Waste prevented vs transfer cost
    cat_waste = redist_result.groupby('category').agg(
        waste_prevented = ('waste_prevented', 'sum'),
        transfer_cost   = ('transfer_cost', 'sum')
    )
    x = range(len(cat_waste))
    width = 0.35
    axes[1].bar([i - width/2 for i in x], cat_waste['waste_prevented'],
                width, label='Waste Prevented (units)', color='#2ECC71', alpha=0.8)
    axes[1].bar([i + width/2 for i in x], cat_waste['transfer_cost'],
                width, label='Transfer Cost (INR)', color='#E74C3C', alpha=0.8)
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(cat_waste.index, rotation=15)
    axes[1].set_title('Waste Prevented vs Transfer Cost')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

### Cell 6 — Run All Optimizers for All Stores
Runs Discount + Replenishment optimizers for all 10 stores and saves results.

In [ ]:
stores = sorted(df['store_id'].unique())
all_discount_results      = []
all_replenishment_results = []

print('Running optimization for all stores...')

for store in stores:
    # Discount Optimizer
    disc_r, disc_status = optimize_discounts(store, df)
    if disc_r is not None:
        all_discount_results.append(disc_r)

    # Replenishment Optimizer
    rep_r, rep_status, cost = optimize_replenishment(store, df, forecast_df)
    all_replenishment_results.append(rep_r)

    print(f'  {store}: Discount={disc_status} | Replenishment={rep_status} | Cost=INR {cost:,.0f}')

# Combine results
all_discounts      = pd.concat(all_discount_results,      ignore_index=True)
all_replenishments = pd.concat(all_replenishment_results, ignore_index=True)

# Save to CSV
all_discounts.to_csv('optimization_discounts.csv',           index=False)
all_replenishments.to_csv('optimization_replenishments.csv', index=False)
redist_result.to_csv('optimization_redistribution.csv',      index=False)

print('\nSaved:')
print('  optimization_discounts.csv')
print('  optimization_replenishments.csv')
print('  optimization_redistribution.csv')
print('\nAll stores optimized!')

### Cell 7 — Combined Optimization Dashboard
Visual summary comparing all stores across all 3 optimization models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Chart 1: Average discount by store ──
store_disc = all_discounts.groupby('store_id')['optimal_discount'].mean().sort_values(ascending=False)
colors_d   = ['#E74C3C' if v >= 20 else '#F39C12' if v >= 10 else '#2ECC71' for v in store_disc.values]
axes[0,0].bar(store_disc.index, store_disc.values, color=colors_d)
axes[0,0].set_xlabel('Store')
axes[0,0].set_ylabel('Avg Optimal Discount %')
axes[0,0].set_title('Average Optimal Discount per Store')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].axhline(y=20, color='red', linestyle='--', alpha=0.5)

# ── Chart 2: Total waste saved by store ──
store_waste = all_discounts.groupby('store_id')['waste_saved_units'].sum().sort_values(ascending=False)
axes[0,1].bar(store_waste.index, store_waste.values, color='#3498DB', alpha=0.8)
axes[0,1].set_xlabel('Store')
axes[0,1].set_ylabel('Units')
axes[0,1].set_title('Total Waste Saved via Discounting (All Stores)')
axes[0,1].tick_params(axis='x', rotation=45)

# ── Chart 3: Total order cost by store ──
store_cost = all_replenishments.groupby('store_id')['order_cost_INR'].sum().sort_values(ascending=False)
axes[1,0].bar(store_cost.index, store_cost.values, color='#9B59B6', alpha=0.8)
axes[1,0].set_xlabel('Store')
axes[1,0].set_ylabel('Total Order Cost (INR)')
axes[1,0].set_title('Optimal Replenishment Cost per Store')
axes[1,0].tick_params(axis='x', rotation=45)

# ── Chart 4: Urgency distribution ──
urgency_counts = all_replenishments.groupby(['store_id', 'urgency']).size().unstack(fill_value=0)
urgency_counts.plot(kind='bar', ax=axes[1,1],
                    color=['#2ECC71', '#F39C12', '#E74C3C'],
                    alpha=0.8)
axes[1,1].set_xlabel('Store')
axes[1,1].set_ylabel('Number of Categories')
axes[1,1].set_title('Replenishment Urgency Distribution')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].legend(title='Urgency')

plt.suptitle('FRESHNESS GUARD — Phase 3 Optimization Dashboard', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nOPTIMIZATION SUMMARY')
print('=' * 50)
print(f"Total waste saved via discounting : {all_discounts['waste_saved_units'].sum():,} units")
print(f"Total replenishment cost          : INR {all_replenishments['order_cost_INR'].sum():,.2f}")
if len(redist_result) > 0:
    print(f"Total waste prevented via redistribution: {redist_result['waste_prevented'].sum():,} units")
print('=' * 50)

### Cell 8 — Natural Language Explanations (Explainable AI)
Converts optimization results into human-readable recommendations.
This is the **Explainable AI** layer your project task requires.

In [ ]:
def generate_optimization_report(store_id, disc_df, rep_df, redist_df):
    """
    Converts optimization numbers into plain English recommendations.
    This is the Explainable AI output for the project.
    """
    report = []
    report.append(f'FRESHNESS GUARD — OPTIMIZATION REPORT: {store_id}')
    report.append('=' * 55)

    # ── Discount recommendations ──
    store_disc = disc_df[disc_df['store_id'] == store_id]
    if len(store_disc) > 0:
        report.append('\nDISCOUNT RECOMMENDATIONS:')
        for _, row in store_disc.iterrows():
            if row['optimal_discount'] >= 20:
                urgency = 'URGENT'
            elif row['optimal_discount'] >= 10:
                urgency = 'MODERATE'
            else:
                urgency = 'LOW'
            report.append(
                f"  [{urgency}] {row['category']}: Apply {row['optimal_discount']}% discount "
                f"(Price: INR {row['original_price']} -> INR {row['new_price']}) "
                f"| Saves {row['waste_saved_units']:,} units from waste"
            )

    # ── Replenishment recommendations ──
    store_rep = rep_df[rep_df['store_id'] == store_id]
    if len(store_rep) > 0:
        report.append('\nREPLENISHMENT RECOMMENDATIONS:')
        for _, row in store_rep.iterrows():
            if row['urgency'] == 'HIGH':
                report.append(
                    f"  [HIGH PRIORITY] Order {row['optimal_order']:,} units of {row['category']} "
                    f"immediately (Lead time: {row['lead_time_days']} days | Cost: INR {row['order_cost_INR']:,.0f})"
                )
            elif row['urgency'] == 'MEDIUM':
                report.append(
                    f"  [MEDIUM] Order {row['optimal_order']:,} units of {row['category']} "
                    f"within 3 days (Cost: INR {row['order_cost_INR']:,.0f})"
                )

    # ── Redistribution recommendations ──
    if len(redist_df) > 0:
        store_redist = redist_df[
            (redist_df['from_store'] == store_id) |
            (redist_df['to_store']   == store_id)
        ]
        if len(store_redist) > 0:
            report.append('\nREDISTRIBUTION RECOMMENDATIONS:')
            for _, row in store_redist.iterrows():
                if row['from_store'] == store_id:
                    report.append(
                        f"  [SEND] Transfer {row['transfer_units']:,} units of {row['category']} "
                        f"to {row['to_store']} | Prevents {row['waste_prevented']:,} units waste"
                    )
                else:
                    report.append(
                        f"  [RECEIVE] Accept {row['transfer_units']:,} units of {row['category']} "
                        f"from {row['from_store']} | Fills stock gap"
                    )

    return '\n'.join(report)

# ── Generate reports for all stores ──
all_reports = {}
for store in stores:
    report = generate_optimization_report(store, all_discounts, all_replenishments, redist_result)
    all_reports[store] = report
    print(report)
    print()

# Save all reports
with open('optimization_reports.txt', 'w', encoding='utf-8') as f:
    for store, report in all_reports.items():
        f.write(report + '\n\n')

print('Saved: optimization_reports.txt')
print('\nPhase 3 Optimization COMPLETE!')
print('Next Step: Phase 4 — Streamlit Dashboard + Simulation')